# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset ([Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution](https://sen.science/doi/10.71728/senscience.qs2f-h81p/)) using the `mlcroissant` library and its Croissant metadata schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Show the metadata information
metadata = dataset.metadata
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
if hasattr(metadata, 'identifier'):
    print(f"Identifier: {metadata.identifier}")
if hasattr(metadata, 'datePublished'):
    print(f"Date Published: {metadata.datePublished}")
if hasattr(metadata, 'keywords'):
    print(f'Sample Keywords: {metadata.keywords[:5] if len(metadata.keywords) > 5 else metadata.keywords}')

## 2. Data Overview
Review available record sets, fields, and their IDs. Each entity is referenced by its unique `@id`.

Let's print all record sets in the dataset along with their fields and columns. This helps in referencing the right `@id`s for extraction.

In [ ]:
# List all available record sets and their structure
if hasattr(dataset.metadata, 'recordSet') and dataset.metadata.recordSet:
    record_sets = dataset.metadata.recordSet
    print(f"Found {len(record_sets)} record set(s):\n")
    all_record_set_ids = []
    for rs in record_sets:
        print(f"Record Set @id: {rs['@id']}")
        all_record_set_ids.append(rs['@id'])
        # Fields info
        if 'field' in rs:
            fields = rs['field']
            print("  Fields:")
            for f in fields:
                if isinstance(f, dict):
                    print(f"    - {f['@id']} (name: {f.get('name', '')}, column: {f.get('column', '')})")
                else:
                    print(f"    - {f}")
        if 'column' in rs:
            columns = rs['column']
            print("  Columns:")
            for c in columns:
                if isinstance(c, dict):
                    print(f"    - {c['@id']} (name: {c.get('name', '')})")
                else:
                    print(f"    - {c}")
        print()
else:
    # If recordSet is not structured as list, get all record sets from dataset via API
    all_record_set_ids = []
    print("Extracting record sets via dataset API:\n")
    # Use internal API to get all record set @id's (if needed)
    # fallback to dataset API
    try:
        # Find all available record sets via .available_record_sets property
        record_set_ids = dataset.available_record_sets
        all_record_set_ids = list(record_set_ids)
        for idx, record_set_id in enumerate(all_record_set_ids):
            print(f"[{idx}] Record Set @id: {record_set_id}")
            try:
                sample = next(dataset.records(record_set=record_set_id))
                print(f"    Sample fields: {list(sample.keys())}")
            except Exception as e:
                print(f"    (Could not print sample: {e})")
            print()
    except Exception as e:
        print("No record sets found via API.")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s obtained above.

We'll load the records for each record set found and preview their content.

In [ ]:
# We'll use the record set IDs detected in the previous step.
# If all_record_set_ids is empty (i.e., no record sets defined in metadata), we'll use a default string based on known FAIR2 examples.

# If the notebook is rerun, this cell remains safe due to protection.
try:
    record_sets = all_record_set_ids
except Exception:
    # fallback if previous cell not executed
    record_sets = []
    print('Warning: No record sets detected, setting to empty.')

if not record_sets:
    # Manually set based on common naming conventions for FAIR2 datasets
    record_sets = [
        'https://sen.science/doi/10.71728/senscience.qs2f-h81p/records/clinical-tabular',
        'https://sen.science/doi/10.71728/senscience.qs2f-h81p/records/variables-tabular',
    ]  # Update as appropriate for dataset structure

dataframes = {}
for record_set_id in record_sets:
    print(f'Loading data for record set: {record_set_id}')
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        if not df.empty:
            print(f'  Fields: {df.columns.tolist()}')
            print(df.head(3))
        else:
            print('  (No records loaded)')
        dataframes[record_set_id] = df
    except Exception as e:
        print(f'  (Failed to load: {e})')

# For exploratory analysis below, pick the main record set (assume clinical-tabular as main)
main_rs = record_sets[0]
print(f"\nPrimary record set for further analysis: {main_rs}")
df = dataframes[main_rs]
print(f"\nFields in {main_rs}: {df.columns.tolist()}")
df.head(5)

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate basic data processing steps:
- Filtering records by a numeric field (e.g., age at diagnosis)
- Normalizing this numeric field
- Grouping data by a categorical variable (e.g., sex, cancer location)

**All fields and columns are referenced by their Croissant `@id`.**

First, let's list the fields, choose relevant numeric and grouping fields, and proceed with operations.

In [ ]:
# Select field @id for numeric and grouping attributes:
# If you don't know the exact @id, list columns first
print("Available columns:")
print(df.columns.tolist())

# For illustration, suppose the field @id for age is 'age_at_second_crc_diagnosis' and sex is 'sex'
# (Update these with actual Croissant @id from your printout if different)

numeric_field_id = 'age_at_second_crc_diagnosis'  # Replace with Croissant @id if different
group_field_id = 'sex'  # Replace with Croissant @id if different

if numeric_field_id not in df.columns or group_field_id not in df.columns:
    print("Please update numeric_field_id and group_field_id to match available columns.")

# Filter out records with age at 2nd CRC diagnosis > threshold
threshold = 60
filtered_df = df[df[numeric_field_id] > threshold]
print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id, group_field_id]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
    filtered_df[numeric_field_id].std()
)
print(f"\nNormalized {numeric_field_id} (mean=0, std=1):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by sex and show average age
if group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'count'])
    print(f"\nMean and count by {group_field_id} (> {threshold}):")
    print(grouped)

## 5. Visualization
Visualize distributions and relationships in the data.

Below, we'll:
- Plot the distribution of age at diagnosis
- Plot average age by sex (or another grouping field)

Please ensure the chosen field @id's are correct!

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field_id], bins=10, kde=True)
plt.title('Distribution of Age at Second CRC Diagnosis')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

# Barplot: mean age by sex
plt.figure(figsize=(6, 4))
sns.barplot(
    x=group_field_id,
    y=numeric_field_id,
    data=df,
    ci=None,
    estimator='mean'
)
plt.title('Mean Age at 2nd CRC Diagnosis by Sex')
plt.ylabel('Mean Age')
plt.show()

## 6. Conclusion

In this notebook, we explored the FAIR² dataset for second primary colorectal cancer using the Croissant schema and `mlcroissant` library:
- Loaded structured metadata and data records by referencing record set and field `@id`.
- Inspected dataset structure, fields, and cohorts.
- Extracted tabular data, filtered, normalized, and grouped by clinically relevant variables (e.g., age at diagnosis, sex).
- Visualized distributions for exploratory purposes.

For further analysis, continue to select relevant clinical, molecular, or pathological variables by their Croissant `@id`, and apply advanced modeling or exploratory techniques. See [mlcroissant documentation](https://mlcommons.github.io/croissant/) for further capabilities.